## Experiment for Two Interventions

#### TODO: Mostar IMAGEM

#### Setting up the pre-conditions

In [52]:
import random
import pandas as pd
random.seed(42)
SAMPLES = 1000
# U, V, W: Cardinality 8
def func_U_V_W():
    """Returns a value for U, V, or W (cardinality 8)."""
    PU = [0.05,0.15,0.10,0.20,0.05,0.25,0.10,0.10]
    PV = [0.30,0.10,0.05,0.15,0.05,0.20,0.10,0.05]
    mylist = [i for i in range(8)]
    U_column = random.choices(mylist, weights = PU, k = SAMPLES)
    V_column = random.choices(mylist, weights = PV, k = SAMPLES)
    W_column = random.choices([0,1], weights = [0.5, 0.5], k = SAMPLES)
    return U_column, V_column, W_column

# A: Cardinality 2
def func_A(parent):
    """Returns a value for A or C (cardinality 2) based on a parent."""
    a = []
    for p in parent:
        a.append(1 if p > 4 else 0)        
    return a

def func_C(parent):
    """Returns a value for A or C (cardinality 2) based on a parent."""
    c = []
    for p in parent:
        c.append(p % 2)
    return c

# B: Cardinality 2
def func_B(parents):
    """Returns a value for B or D (cardinality 2) based on its parents."""
    b = []
    for i in range(SAMPLES):
            b.append((parents[0][i]+parents[1][i]) % 2)
    return b

# D: Cardinality 2
def func_D(parents):
    lookup_table = {
        (0, 0): 1,
        (0, 1): 0,
        (1, 0): 1,
        (1, 1): 1,
        (2, 0): 1,
        (2, 1): 1,
        (3, 0): 1,
        (3, 1): 0,
        (4, 0): 1,
        (4, 1): 0,
        (5, 0): 0,
        (5, 1): 1,
        (6, 0): 1,
        (6, 1): 0,
        (7, 0): 1,
        (7, 1): 0
    }
    d = []
    for i in range(SAMPLES):
            d.append(lookup_table[(parents[0][i],parents[1][i])])
    return d


# TODO: Adicionar ruído W
# Y: Cardinality 2
def func_Y(parents):
    """Returns a value for Y (cardinality 2) based on its parents."""
    y = []
    for i in range(SAMPLES):
            y.append(1 if (parents[0][i] == 1) and (parents[1][i] == 1 and parents[2][i]) else 0)
    return y

# Example to evaluate the functions
# Assign values to U, V, and W (randomly chosen here)
U, V, W = func_U_V_W()
U_cardinality = 8
V_cardinality = 8

# Compute A and C
A = func_A(U)
C = func_C(V)

# Compute B and D
B = func_B([U, A])
D = func_D([V, C])

# Compute Y
Y = func_Y([B, D, W])

df = pd.DataFrame({
    "A": A,
    "B": B,
    "C": C,
    "D": D,
    "Y": Y,
})


df.to_csv("total_two_interventions.csv", index=False)
# row_frequencies = df.groupby(df.columns.tolist()).size().reset_index(name='Frequency')
# row_frequencies['Prob'] = row_frequencies['Frequency'] / SAMPLES
# row_frequencies.drop(columns='Frequency')
# row_frequencies.drop(columns='Frequency').to_csv("prob_two_interventions.csv", index=False)
# p =0
# print(row_frequencies)
# for index, row in row_frequencies.iterrows():
#     p += row['Prob']
# print(p)

In [53]:
df.head()

,A,B,C,D,Y
0,1,0,0,1,0
1,0,0,1,1,0
2,0,0,1,0,0
3,0,0,1,0,0
4,1,0,1,1,0


In [54]:
row_frequencies = df.groupby(df.columns.tolist()).size().reset_index(name='Frequency')
# row_frequencies['Prob'] = row_frequencies['Frequency'] / SAMPLES
# row_frequencies.drop(columns='Frequency')
row_frequencies.to_csv("freq_two_interventions.csv", index=False)
# p =0
# print(row_frequencies)
# for index, row in row_frequencies.iterrows():
#     p += row['Prob']
# print(p)
row_frequencies


,A,B,C,D,Y,Frequency
0,0,0,0,1,0,96
1,0,0,1,0,0,48
2,0,0,1,1,0,66
3,0,1,0,1,0,75
4,0,1,0,1,1,74
5,0,1,1,0,0,59
6,0,1,1,1,0,54
7,0,1,1,1,1,60
8,1,0,0,1,0,174
9,1,0,1,0,0,76


### Generate Mechanisms

In [55]:
from causal_solver.Helper import Helper
from partition_methods.relaxed_problem.python.graph import Graph


test_name = "duas_intervencoes"
print(f"Please, enter the graph in the default format")
dag: Graph = Graph.parse(fromInterface=False,file_path=f"/home/lawand/Canonical-Partition/test_cases/inputs/{test_name}.txt")
print("Please, enter a path for the csv")
csvPath = f"{test_name}.csv"
print(f"For an inference P(Y=y|do(X=x)) please, enter, in this order: X, x, Y, y")
valueIntervention = 1
interventionVariableLabel, interventionVariableValue, targetVariableLabel, targetVariableValue = f"A {valueIntervention} Y 1".split()
interventionVariable      = dag.labelToIndex[interventionVariableLabel]
targetVariable            = dag.labelToIndex[targetVariableLabel]
interventionVariableValue = int(interventionVariableValue)
targetVariableValue       = int(targetVariableValue)

def update_mechanisms(mecanisms, endo_var, exo_var):
    new_mecanisms = []
    i = 0
    for m in mecanisms:
        new_mec = {}
        for key, value in m.items():
            if key != '':
                n_key = dag.indexToLabel[int(key[0])] +key[1:len(key)]
                new_mec[n_key] = f"{endo_var}={value}"
            else :
                new_mec[key] = value
        new_mec[f"{exo_var}"] = f"{i}"
        new_mecanisms.append(new_mec)
        i+=1
    return new_mecanisms

_,_,mecanisms = Helper.mechanisms_generator(latentNode=dag.labelToIndex["U"], endogenousNodes=[dag.labelToIndex["A"],dag.labelToIndex["B"]],
                                        cardinalities=dag.cardinalities,graphNodes=dag.graphNodes,v=False)
u_mecanisms = update_mechanisms(mecanisms, "B", "U")
print("**************************************************************")
print("Mecanismos de U:")
print(u_mecanisms)
_,_,mecanisms = Helper.mechanisms_generator(latentNode=dag.labelToIndex["V"], endogenousNodes=[dag.labelToIndex["C"],dag.labelToIndex["D"]],
                                        cardinalities=dag.cardinalities,graphNodes=dag.graphNodes,v=False)
v_mecanisms = update_mechanisms(mecanisms, "D", "V")

print("**************************************************************")
print("Mecanismos de V:")
print(v_mecanisms)


Please, enter the graph in the default format
Please, enter a path for the csv
For an inference P(Y=y|do(X=x)) please, enter, in this order: X, x, Y, y
**************************************************************
Mecanismos de U:
[{'': 0, 'A=0': 'B=0', 'A=1': 'B=0', 'U': '0'}, {'': 0, 'A=0': 'B=0', 'A=1': 'B=1', 'U': '1'}, {'': 0, 'A=0': 'B=1', 'A=1': 'B=0', 'U': '2'}, {'': 0, 'A=0': 'B=1', 'A=1': 'B=1', 'U': '3'}, {'': 1, 'A=0': 'B=0', 'A=1': 'B=0', 'U': '4'}, {'': 1, 'A=0': 'B=0', 'A=1': 'B=1', 'U': '5'}, {'': 1, 'A=0': 'B=1', 'A=1': 'B=0', 'U': '6'}, {'': 1, 'A=0': 'B=1', 'A=1': 'B=1', 'U': '7'}]
**************************************************************
Mecanismos de V:
[{'': 0, 'C=0': 'D=0', 'C=1': 'D=0', 'V': '0'}, {'': 0, 'C=0': 'D=0', 'C=1': 'D=1', 'V': '1'}, {'': 0, 'C=0': 'D=1', 'C=1': 'D=0', 'V': '2'}, {'': 0, 'C=0': 'D=1', 'C=1': 'D=1', 'V': '3'}, {'': 1, 'C=0': 'D=0', 'C=1': 'D=0', 'V': '4'}, {'': 1, 'C=0': 'D=0', 'C=1': 'D=1', 'V': '5'}, {'': 1, 'C=0': 'D=1', 'C=1':

#### What are the interventions?

In [56]:
doA = 1
doC = 1

#### Creating Gurobi's Model

In [57]:
from gurobipy import Model, GRB

model = Model("Multilinear_Optimization")
number_of_vars = U_cardinality + V_cardinality
vars = model.addVars(number_of_vars, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="vars")

### Função Objetivo e Restrições:

#### Função Objetivo:
$$ P(y|do(a),do(c)) = \sum_{b,d}P(y|B,D,do(c),do(a))P(B|do(a))P(D|do(c)) \\
= \sum_{b,d,u,v}\underbrace{P(y|B,D)}_{\mathrm{dados}}\underbrace{P(B|a,U)P(D|c,V)}_{\mathrm{mecanismos}}\underbrace{P(U)P(V)}_{\mathrm{variaveis \ de \ otimizacao(bilinear)}}$$

#### Restrições:
$$ P(a,b) = \sum_{u}\underbrace{P(a,b|U)}_{\mathrm{mecanismo}}\underbrace{P(U)}_{\mathrm{variavel}} \\$$
$$ P(c,d) = \sum_{v}\underbrace{P(c,d|V)}_{\mathrm{mecanismo}}\underbrace{P(V)}_{\mathrm{variavel}}$$


In [66]:
# Func Objetivo

def verify_a_b_u_mechanism(b, u):
    for key, value in u_mecanisms[u].items():
        if "A" in key and str(doA) in key and str(b) in value:
            return True
    return False

def verify_c_d_v_mechanism(d, v):
    for key, value in v_mecanisms[v].items():
        if "C" in key and str(doC) in key and str(d) in value:
            return True
    return False

def get_p_y_given_B_D(b, d):
    number_y_given_b_d = 0
    total_of_b_d = 0
    for index, row in row_frequencies.iterrows():
        if row["B"] == b and row["D"] == d:
            total_of_b_d+=row["Frequency"]
            if row["Y"] == 1:
                number_y_given_b_d += row["Frequency"]
    return number_y_given_b_d/total_of_b_d

def set_objective_func(vars):
    expression = 0
    for b in range(2):
        for d in range(2):
            for u in range(U_cardinality):
                for v in range(V_cardinality):
                    if ((not verify_a_b_u_mechanism(b, u)) or (not verify_c_d_v_mechanism(d, v))):
                        continue
                    p_y_given_B_D = get_p_y_given_B_D(b, d)
                    expression += (p_y_given_B_D*vars[u]*vars[v+8])
    return expression
model.setObjective(set_objective_func(vars), GRB.MAXIMIZE)


0.0 + [ 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gurobi.Var *Awaiting Model Update*> * <gurobi.Var *Awaiting Model Update*> + 0.0 <gur

In [6]:
# Restrições
def set_constraints():
    pass
def get_conditional_prob_a_b_given_U():
    pass     
def get_conditional_prob_c_d_given_V():
    pass

In [7]:
from gurobipy import Model, GRB

def optm(func_obj: list[int], eq_restr: list[list[int]], valores_das_restr: list[int]):
    model = Model("Multilinear_Optimization")

    n_vars = len(func_obj)
    vars = model.addVars(n_vars, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="vars")

    n_rows = len(eq_restr)
    n_columns = n_vars
    for i in range(n_rows):
        expression = 0
        for j in range(n_columns):
            expression += (eq_restr[i][j]*vars[i])
        model.addConstr(valores_das_restr[i] <= expression, name=f"constraint_{i}_lower")
        model.addConstr(expression <= valores_das_restr[i], name=f"constraint_{i}_upper")

    # TODO: Set objective function

    # Set the objective to minimize the sum of bilinear terms
    model.setObjective(xy + yz + xz, GRB.MINIMIZE)

    # Solve the model
    model.optimize()

    # Print the results
    if model.status == GRB.OPTIMAL:
        print(f"Optimal solution found:")
        print(f"x = {x.X}")
        print(f"y = {y.X}")
        print(f"z = {z.X}")
        print(f"Objective value = {model.ObjVal}")
    else:
        print("No optimal solution found.")
